# Silver — ERP Customer Location
Country per customer from the ERP.

`bronze.erp_loc_a101` → `silver.erp_customer_location`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_loc_a101")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Clean customer id
Ids arrive as `AW-00011000`; remove the dash to match the CRM format.

In [ ]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

### Normalize country
Only known codes are mapped; other countries are already spelled out, so keep them as-is.

In [ ]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customer_location")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10;